In [44]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

In [45]:
scenarios = ["Electrification", "Hydrogen", "Ammonia", "Methanol"]

In [46]:
results_path = os.path.join("..", "Outputs")

In [47]:
country_code = "SK"
# ['EU27', 'AT', 'BE', 'BG', 'CY', 
# 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 
# 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL', 
# 'PT', 'RO', 'SE', 'SI', 'SK']


In [48]:
base_df = pd.read_excel(os.path.join(results_path, f"PtX_demand_{country_code}.xlsx"))

In [49]:
base_df[["FuelGroup","Year","Pass Road"]]

,FuelGroup,Year,Pass Road
0,Power,2030,4.084074e-03
1,Hydrogen,2030,1.895260e-04
2,Methanol,2030,NaN
3,Ammonia,2030,NaN
4,Synthetic Gases,2030,NaN
5,Biogenic Gases,2030,6.470281e-05
6,Fossil Gases,2030,7.299738e-04
7,Synthetic Liquids,2030,NaN
8,Biogenic Liquids,2030,3.515587e-03
9,Fossil Liquids,2030,3.388154e-02


In [50]:
column_order = [
    "Iron & steel", "Chemicals", "Non-metallic minerals",  # Industry
    "Pass Road", "Pass Rail", "Pass Aviation",             # Transport (passenger)
    "Freight Road", "Freight Rail",                        # Transport (freight)
    "Maritime"                                             # Maritime (bunkers)
]

### Define targets for 2050 per scenario

In [51]:
scenario_targets = {
    "Electrification": {
        "Power": {
            "Iron & steel": 0.40,
            "Chemicals": 0.35,
            "Non-metallic minerals": 0.55,
            "Pass Road": 0.95,
            "Pass Rail": 0.98,
            "Freight Road": 0.85,
            "Freight Rail": 0.95,
            "Maritime national": 0.80,
            "Maritime international": 0.30
        },

        "Hydrogen": {
            "Iron & steel": 0.40,
            "Chemicals": 0.25,
            "Non-metallic minerals": 0.25,
            "Pass Road": 0.02,
            "Freight Road": 0.10,
            "Maritime national": 0.05,
            "Maritime international": 0.05
        },

        "Methanol": {
            "Chemicals": 0.15,
            "Maritime national": 0.10,
            "Maritime international": 0.30
        },

        "Ammonia": {
            "Chemicals": 0.05,
            "Maritime international": 0.30
        },

        "Synthetic Liquids": {
            "Pass Road": 0.90
        },

        "Renewable Energy Carrier": {
            "Iron & steel": 0.20,
            "Chemicals": 0.20,
            "Non-metallic minerals": 0.20,
            "Pass Road": 0.03,
            "Pass Rail": 0.02,
            "Pass Aviation": 0.10,
            "Freight Road": 0.05,
            "Freight Rail": 0.05,
            "Maritime national": 0.05,
            "Maritime international": 0.05
        },
    }
}

In [ ]:
electrification_targets = {
    "Pass Road": {
        "Power": 0.95,
        "Hydrogen": 0.02,
        "Renewable Energy Carrier": 0.03
    },

    "Pass Rail": {
        "Power": 0.98,
        "Renewable Energy Carrier": 0.02
    },

    "Freight Road": {
        "Power": 0.85,
        "Hydrogen": 0.10,
        "Renewable Energy Carrier": 0.05
    },

    "Freight Rail": {
        "Power": 0.95,
        "Renewable Energy Carrier": 0.05
    }
}

### Ramp-up function and definition of country specific threshold 

In [53]:
def get_sector_shares(df, sector, year):
    df_year = df[df["Year"] == year].copy()

    overall = df_year.loc[df_year["FuelGroup"] == "Overall Demand", sector].iloc[0]

    fuels = df_year[df_year["FuelGroup"] != "Overall Demand"][["FuelGroup", sector]].copy()

    fuels[sector] = fuels[sector].fillna(0)
    fuels["Share"] = fuels[sector] / overall

    return fuels[["FuelGroup", "Share"]]

In [54]:
def exponential_path(start, target, year,start_year=2030, end_year=2050, k=0.12):

    if year <= start_year:
        return start

    if year >= end_year:
        return target

    t = year - start_year
    T = end_year - start_year
    w = (1 - np.exp(-k * t)) / (1 - np.exp(-k * T))

    return start + w * (target - start)

In [57]:
def build_sector_scenario(df, sector, years):
    base = get_sector_shares(df, sector, 2030)
    start = dict(zip(base["FuelGroup"], base["Share"]))
    targets = electrification_targets[sector]

    all_fuels = set(start.keys()).union(set(targets.keys()))
    rows = []
    for y in years:
        row = {}

        for f in all_fuels:
            s0 = start.get(f, 0.0)
            s1 = targets.get(f, 0.0)
            row[f] = exponential_path(s0, s1, y)

        # normalize
        total = sum(row.values())

        for f in row:
            row[f] /= total

        row["Year"] = y
        rows.append(row)
    return pd.DataFrame(rows)

In [58]:
years = [2030, 2035, 2040, 2045, 2050]

df_pass_road = build_sector_scenario(
    base_df,
    "Pass Road",
    years
)

print(df_pass_road.round(3))

   Fossil Residuals  Ammonia  Fossil Liquids  Biogenic Gases  Fossil Gases  \
0               0.0      0.0           0.798           0.002         0.017   
1               0.0      0.0           0.402           0.001         0.009   
2               0.0      0.0           0.185           0.000         0.004   
3               0.0      0.0           0.065           0.000         0.001   
4               0.0      0.0           0.000           0.000         0.000   

   Power  Hydrogen  Synthetic Gases  Biogenic Liquids  Methanol  \
0  0.096     0.004              0.0             0.083       0.0   
1  0.520     0.012              0.0             0.042       0.0   
2  0.752     0.016              0.0             0.019       0.0   
3  0.880     0.019              0.0             0.007       0.0   
4  0.950     0.020              0.0             0.000       0.0   

   Biomass [Solid]  Renewable Energy Carrier  Synthetic Liquids  Year  
0              0.0                     0.000            

### Visualisation for ramp-up curves